In [1]:
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import nltk

from scipy.sparse import vstack

from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
SEED = 40

TRAIN_PATH = Path("corpus/train.jsonl")
VALIDATION_PATH = Path("corpus/validation.jsonl")
TEST_PATH = Path("corpus/test.jsonl")

NON_PUN_LABEL = 0
PUN_LABEL = 1

random.seed(SEED)
np.random.seed(SEED)

In [3]:
def load_jsonl(file_path):
    rows = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                rows.append(json.loads(line))

    return pd.DataFrame(rows)


train_df = load_jsonl(TRAIN_PATH)
validation_df = load_jsonl(VALIDATION_PATH)
test_df = load_jsonl(TEST_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(validation_df.head())
display(test_df.head())

Train shape: (3990, 5)
Validation shape: (570, 5)
Test shape: (1140, 5)


,id,text,label,tokens,labels
0,5.792.H,Por que a mulher esotérica não conseguia engra...,1,"[Por, que, a, mulher, esotérica, não, consegui...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]"
1,5.733.H,Qual o sambista passou a dar presente pra todo...,1,"[Qual, o, sambista, passou, a, dar, presente, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,4.652.N,Um homem matou uma ovelha e agora foi preso . ...,0,"[Um, homem, matou, uma, ovelha, e, agora, foi,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,5.2585.N,Qual apresentador de TV vive gripado? Fausto S...,0,"[Qual, apresentador, de, TV, vive, gripado, ?,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,5.28.N,Qual é a modelo mais bela que existe? Gisele B...,0,"[Qual, é, a, modelo, mais, bela, que, existe, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


,id,text,label,tokens,labels
0,5.46.H,Por que o carteiro foi à feira? Porque tinha u...,1,"[Por, que, o, carteiro, foi, à, feira, ?, Porq...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0]"
1,5.1811.H,Qual é o animal que está sempre cansado? Dorme...,1,"[Qual, é, o, animal, que, está, sempre, cansad...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]"
2,5.1990.N,Uma cerveja se associou a um comediante para a...,0,"[Uma, cerveja, se, associou, a, um, comediante...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,4.103.H,Qual é a única coisa que se faz sempre em nume...,1,"[Qual, é, a, única, coisa, que, se, faz, sempr...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0]"
4,4.19.H,O meu baralho de cartas fica doido quando ligo...,1,"[O, meu, baralho, de, cartas, fica, doido, qua...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


,id,text,label,tokens,labels
0,4.66.N,Eu adoro fiambre . E também queijo.,0,"[Eu, adoro, fiambre, ., E, também, queijo, .]","[0, 0, 0, 0, 0, 0, 0, 0]"
1,5.2933.N,Qual a moeda inacreditável? Euro,0,"[Qual, a, moeda, inacreditável, ?, Euro]","[0, 0, 0, 0, 0, 0]"
2,5.3868.N,Qual marca de hotéis virou presidente? Trump.,0,"[Qual, marca, de, hotéis, virou, presidente, ?...","[0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,5.3612.N,Qual é a loja que fez um estande em volta do d...,0,"[Qual, é, a, loja, que, fez, um, estande, em, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,5.1714.H,Você gosta de Katy Perry? Katy perguntou?,1,"[Você, gosta, de, Katy, Perry, ?, Katy, pergun...","[0, 0, 0, 0, 0, 0, 1, 1, 0]"


In [4]:
required_columns = {"id", "text", "label"}

for split_name, dataframe in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df)
]:
    missing_columns = required_columns - set(dataframe.columns)

    if missing_columns:
        raise ValueError(f"Missing columns in {split_name}: {missing_columns}")

    dataframe["id"] = dataframe["id"].astype(str)
    dataframe["text"] = dataframe["text"].astype(str)
    dataframe["label"] = dataframe["label"].astype(int)

    invalid_labels = set(dataframe["label"].unique()) - {0, 1}

    if invalid_labels:
        raise ValueError(f"Invalid labels in {split_name}: {invalid_labels}")

print("Train label distribution:")
display(train_df["label"].value_counts().sort_index())

print("Validation label distribution:")
display(validation_df["label"].value_counts().sort_index())

print("Test label distribution:")
display(test_df["label"].value_counts().sort_index())

Train label distribution:


,count
label,
0,1995
1,1995


Validation label distribution:


,count
label,
0,285
1,285


Test label distribution:


,count
label,
0,570
1,570


In [5]:
def extract_pair_id(example_id):
    return re.sub(r"\.[HN]$", "", str(example_id))


def extract_pair_suffix(example_id):
    match = re.search(r"\.([HN])$", str(example_id))

    if match:
        return match.group(1)

    return None


for dataframe in [train_df, validation_df, test_df]:
    dataframe["pair_id"] = dataframe["id"].apply(extract_pair_id)
    dataframe["pair_suffix"] = dataframe["id"].apply(extract_pair_suffix)


print("Train suffix vs label:")
display(pd.crosstab(train_df["pair_suffix"], train_df["label"]))

print("Validation suffix vs label:")
display(pd.crosstab(validation_df["pair_suffix"], validation_df["label"]))

print("Test suffix vs label:")
display(pd.crosstab(test_df["pair_suffix"], test_df["label"]))

Train suffix vs label:


label,0,1
pair_suffix,,
H,0,1995
N,1995,0


Validation suffix vs label:


label,0,1
pair_suffix,,
H,0,285
N,285,0


Test suffix vs label:


label,0,1
pair_suffix,,
H,0,570
N,570,0


In [6]:
def validate_complete_pairs(dataframe, split_name):
    pair_sizes = dataframe.groupby("pair_id").size()
    invalid_size_pairs = pair_sizes[pair_sizes != 2]

    if len(invalid_size_pairs) > 0:
        display(invalid_size_pairs.head(20))
        raise ValueError(
            f"{split_name}: {len(invalid_size_pairs)} pairs do not have exactly 2 examples."
        )

    invalid_label_pairs = []

    for pair_id, group in dataframe.groupby("pair_id"):
        labels = set(group["label"].tolist())

        if labels != {NON_PUN_LABEL, PUN_LABEL}:
            invalid_label_pairs.append(pair_id)

    if len(invalid_label_pairs) > 0:
        raise ValueError(
            f"{split_name}: {len(invalid_label_pairs)} pairs do not have exactly one 0 and one 1."
        )

    print(f"{split_name}: valid pairs.")
    print(f"{split_name}: {len(pair_sizes)} pairs, {len(dataframe)} examples.")


validate_complete_pairs(train_df, "Train")
validate_complete_pairs(validation_df, "Validation")
validate_complete_pairs(test_df, "Test")

train_pair_ids = set(train_df["pair_id"])
validation_pair_ids = set(validation_df["pair_id"])
test_pair_ids = set(test_df["pair_id"])

train_validation_overlap = train_pair_ids.intersection(validation_pair_ids)
train_test_overlap = train_pair_ids.intersection(test_pair_ids)
validation_test_overlap = validation_pair_ids.intersection(test_pair_ids)

print("Train/validation pair overlap:", len(train_validation_overlap))
print("Train/test pair overlap:", len(train_test_overlap))
print("Validation/test pair overlap:", len(validation_test_overlap))

if train_validation_overlap or train_test_overlap or validation_test_overlap:
    raise ValueError("There is pair_id overlap between splits.")

print("OK: no pair_id overlap between train, validation and test.")

Train: valid pairs.
Train: 1995 pairs, 3990 examples.
Validation: valid pairs.
Validation: 285 pairs, 570 examples.
Test: valid pairs.
Test: 570 pairs, 1140 examples.
Train/validation pair overlap: 0
Train/test pair overlap: 0
Validation/test pair overlap: 0
OK: no pair_id overlap between train, validation and test.


In [7]:
def build_unordered_pair_dataframe(dataframe):
    pair_rows = []

    for pair_id, group in dataframe.groupby("pair_id"):
        group = group.sort_values("id").reset_index(drop=True)

        if len(group) != 2:
            continue

        row_a = group.iloc[0]
        row_b = group.iloc[1]

        pair_rows.append({
            "pair_id": pair_id,

            "a_id": row_a["id"],
            "a_text": row_a["text"],
            "a_label": int(row_a["label"]),

            "b_id": row_b["id"],
            "b_text": row_b["text"],
            "b_label": int(row_b["label"])
        })

    return pd.DataFrame(pair_rows)


train_pairs_df = build_unordered_pair_dataframe(train_df)
validation_pairs_df = build_unordered_pair_dataframe(validation_df)
test_pairs_df = build_unordered_pair_dataframe(test_df)

print("Train pairs:", train_pairs_df.shape)
print("Validation pairs:", validation_pairs_df.shape)
print("Test pairs:", test_pairs_df.shape)

display(train_pairs_df.head())

Train pairs: (1995, 7)
Validation pairs: (285, 7)
Test pairs: (570, 7)


,pair_id,a_id,a_text,a_label,b_id,b_text,b_label
0,1.1,1.1.H,Deve ser difícil ser professor de natação. Voc...,1,1.1.N,Deve ser difícil ser professor de natação . Vo...,0
1,1.4,1.4.H,O que uma impressora falou para a outra? Essa ...,1,1.4.N,O que uma impressora falou para a outra? Essa ...,0
2,1.5,1.5.H,Por que a galinha bateu a cabeça contra a pare...,1,1.5.N,Por que a galinha bateu a cabeça contra a pare...,0
3,2.10,2.10.H,Como o padre bateu o carro? Dando uma rezinha.,1,2.10.N,Como o padre bateu o carro? Dando uma ré.,0
4,2.12,2.12.H,Por que a vaca foi para o espaço? Para se enco...,1,2.12.N,Por que a vaca foi para o espaço? Para se enco...,0


In [8]:
feature_extractor = FeatureUnion([
    (
        "word_tfidf",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 3),
            sublinear_tf=True,
            min_df=1,
            max_df=0.95,
            lowercase=True
        )
    ),
    (
        "char_tfidf",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            sublinear_tf=True,
            min_df=1,
            max_df=0.95,
            lowercase=True
        )
    )
])

feature_extractor

FeatureUnion(transformer_list=[('word_tfidf',
                                TfidfVectorizer(max_df=0.95, ngram_range=(1, 3),
                                                sublinear_tf=True)),
                               ('char_tfidf',
                                TfidfVectorizer(analyzer='char_wb', max_df=0.95,
                                                ngram_range=(3, 5),
                                                sublinear_tf=True))])

In [9]:
feature_extractor.fit(train_df["text"])

print("Feature extractor fitted.")

Feature extractor fitted.


In [10]:
def build_pair_difference_dataset(pair_dataframe, feature_extractor):
    a_vectors = feature_extractor.transform(pair_dataframe["a_text"])
    b_vectors = feature_extractor.transform(pair_dataframe["b_text"])

    x_forward = a_vectors - b_vectors
    y_forward = pair_dataframe["a_label"].astype(int).to_numpy()

    x_backward = b_vectors - a_vectors
    y_backward = pair_dataframe["b_label"].astype(int).to_numpy()

    x_pair = vstack([x_forward, x_backward])
    y_pair = np.concatenate([y_forward, y_backward])

    return x_pair, y_pair


x_train_pair, y_train_pair = build_pair_difference_dataset(
    train_pairs_df,
    feature_extractor
)

x_validation_pair, y_validation_pair = build_pair_difference_dataset(
    validation_pairs_df,
    feature_extractor
)

print("Pair-difference train shape:", x_train_pair.shape)
print("Pair-difference validation shape:", x_validation_pair.shape)

print("Train pair labels:")
display(pd.Series(y_train_pair).value_counts().sort_index())

print("Validation pair labels:")
display(pd.Series(y_validation_pair).value_counts().sort_index())

Pair-difference train shape: (3990, 96427)
Pair-difference validation shape: (570, 96427)
Train pair labels:


,count
0,1995
1,1995


Validation pair labels:


,count
0,285
1,285


In [11]:
lr_l2_model = LogisticRegression(
    random_state=SEED,
    max_iter=5000,
    C=1.0,
    class_weight="balanced",
    solver="liblinear"
)

lr_l1_model = LogisticRegression(
    random_state=SEED,
    max_iter=5000,
    C=0.5,
    penalty="l1",
    class_weight="balanced",
    solver="liblinear"
)

sgd_model = SGDClassifier(
    loss="log_loss",
    penalty="elasticnet",
    alpha=1e-4,
    l1_ratio=0.15,
    class_weight="balanced",
    random_state=SEED,
    max_iter=3000,
    tol=1e-4
)

svm_model = SVC(
    kernel="linear",
    C=1.0,
    probability=True,
    class_weight="balanced",
    random_state=SEED
)

pairwise_ensemble_model = VotingClassifier(
    estimators=[
        ("lr_l2", lr_l2_model),
        ("lr_l1", lr_l1_model),
        ("sgd", sgd_model),
        ("svm", svm_model)
    ],
    voting="soft",
    weights=[3, 2, 2, 2],
    n_jobs=1
)

pairwise_ensemble_model

VotingClassifier(estimators=[('lr_l2',
                              LogisticRegression(class_weight='balanced',
                                                 max_iter=5000, random_state=40,
                                                 solver='liblinear')),
                             ('lr_l1',
                              LogisticRegression(C=0.5, class_weight='balanced',
                                                 max_iter=5000, penalty='l1',
                                                 random_state=40,
                                                 solver='liblinear')),
                             ('sgd',
                              SGDClassifier(class_weight='balanced',
                                            loss='log_loss', max_iter=3000,
                                            penalty='elasticnet',
                                            random_state=40, tol=0.0001)),
                             ('svm',
                              SVC(class_weight='balanced', kernel='linear',
                                  probability=True, random_state=40))],
                 n_jobs=1, voting='soft', weights=[3, 2, 2, 2])

In [12]:
pairwise_ensemble_model.fit(x_train_pair, y_train_pair)

print("Pairwise ensemble training finished.")

Pairwise ensemble training finished.


In [13]:
def build_single_pair_comparison_matrix(pair_dataframe, feature_extractor):
    a_vectors = feature_extractor.transform(pair_dataframe["a_text"])
    b_vectors = feature_extractor.transform(pair_dataframe["b_text"])

    x_pair_comparison = a_vectors - b_vectors
    y_first_is_pun = pair_dataframe["a_label"].astype(int).to_numpy()

    return x_pair_comparison, y_first_is_pun


x_validation_comparison, y_validation_first_is_pun = build_single_pair_comparison_matrix(
    validation_pairs_df,
    feature_extractor
)

validation_proba_first_is_pun = pairwise_ensemble_model.predict_proba(
    x_validation_comparison
)[:, 1]

validation_pair_predictions_df = validation_pairs_df.copy()
validation_pair_predictions_df["prob_a_is_pun"] = validation_proba_first_is_pun

display(validation_pair_predictions_df.head())

,pair_id,a_id,a_text,a_label,b_id,b_text,b_label,prob_a_is_pun
0,1.6,1.6.H,Você sabe qual é a montanha mais limpa de toda...,1,1.6.N,Você sabe qual é a montanha mais alta de todas...,0,0.854342
1,2.6,2.6.H,Por que o bombeiro não gosta de andar? Porque ...,1,2.6.N,Por que o bombeiro não gosta de andar? Porque ...,0,0.599898
2,3.6,3.6.H,Por que o policial não gosta de sabão? Porque ...,1,3.6.N,Por que o policial não gosta de sabão? Porque ...,0,0.684840
3,4.103,4.103.H,Qual é a única coisa que se faz sempre em nume...,1,4.103.N,Qual é a única coisa que se faz sempre em nume...,0,0.452369
4,4.104,4.104.H,Qual é a consola de jogos preferida dos políci...,1,4.104.N,Qual é a consola de jogos preferida dos políci...,0,0.771441


In [14]:
def find_best_pair_threshold(y_true_first_is_pun, probabilities, metric_name="accuracy"):
    thresholds = np.linspace(
        probabilities.min(),
        probabilities.max(),
        300
    )

    best_threshold = 0.5
    best_value = -1.0

    for threshold in thresholds:
        y_pred = (probabilities > threshold).astype(int)

        if metric_name == "accuracy":
            value = accuracy_score(y_true_first_is_pun, y_pred)
        elif metric_name == "f1_macro":
            value = f1_score(
                y_true_first_is_pun,
                y_pred,
                average="macro",
                zero_division=0
            )
        else:
            raise ValueError("metric_name must be 'accuracy' or 'f1_macro'.")

        if value > best_value:
            best_value = value
            best_threshold = threshold

    return best_threshold, best_value


best_pair_threshold, best_validation_pair_accuracy = find_best_pair_threshold(
    y_validation_first_is_pun,
    validation_proba_first_is_pun,
    metric_name="accuracy"
)

print("Best pair threshold:", best_pair_threshold)
print("Best validation pair accuracy:", f"{best_validation_pair_accuracy:.4f}")

Best pair threshold: 0.23690298354428585
Best validation pair accuracy: 0.9965


In [15]:
def predict_items_from_pair_model(
    pair_dataframe,
    feature_extractor,
    pairwise_model,
    threshold
):
    x_pair_comparison, y_first_is_pun = build_single_pair_comparison_matrix(
        pair_dataframe,
        feature_extractor
    )

    probabilities = pairwise_model.predict_proba(x_pair_comparison)[:, 1]
    first_is_pun_predictions = (probabilities > threshold).astype(int)

    item_rows = []

    for index, row in pair_dataframe.reset_index(drop=True).iterrows():
        first_is_pun = int(first_is_pun_predictions[index])

        if first_is_pun == 1:
            a_prediction = PUN_LABEL
            b_prediction = NON_PUN_LABEL
        else:
            a_prediction = NON_PUN_LABEL
            b_prediction = PUN_LABEL

        item_rows.append({
            "pair_id": row["pair_id"],
            "id": row["a_id"],
            "text": row["a_text"],
            "label": int(row["a_label"]),
            "prediction": a_prediction,
            "prob_a_is_pun": probabilities[index]
        })

        item_rows.append({
            "pair_id": row["pair_id"],
            "id": row["b_id"],
            "text": row["b_text"],
            "label": int(row["b_label"]),
            "prediction": b_prediction,
            "prob_a_is_pun": probabilities[index]
        })

    predictions_df = pd.DataFrame(item_rows)

    return predictions_df

In [16]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "recall_macro": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "f1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "precision_weighted": precision_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "recall_weighted": recall_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "f1_weighted": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )
    }


def compute_pair_metrics(predictions_df):
    pair_exact_results = []

    for pair_id, group in predictions_df.groupby("pair_id"):
        if len(group) != 2:
            continue

        pair_exact_correct = bool(
            (group["label"] == group["prediction"]).all()
        )

        pair_exact_results.append(pair_exact_correct)

    return {
        "pair_exact_accuracy": float(np.mean(pair_exact_results)) if pair_exact_results else 0.0,
        "evaluated_pairs": int(len(pair_exact_results))
    }


def print_metrics(title, metrics):
    print(title)
    print("=" * len(title))

    for metric_name, metric_value in metrics.items():
        if isinstance(metric_value, float):
            print(f"{metric_name}: {metric_value:.4f}")
        else:
            print(f"{metric_name}: {metric_value}")

In [17]:
validation_predictions_df = predict_items_from_pair_model(
    pair_dataframe=validation_pairs_df,
    feature_extractor=feature_extractor,
    pairwise_model=pairwise_ensemble_model,
    threshold=best_pair_threshold
)

validation_metrics = compute_metrics(
    validation_predictions_df["label"],
    validation_predictions_df["prediction"]
)

validation_metrics.update(
    compute_pair_metrics(validation_predictions_df)
)

print_metrics("Validation results - pairwise ensemble", validation_metrics)

display(validation_predictions_df.head())

Validation results - pairwise ensemble
accuracy: 0.9965
precision_macro: 0.9965
recall_macro: 0.9965
f1_macro: 0.9965
precision_weighted: 0.9965
recall_weighted: 0.9965
f1_weighted: 0.9965
pair_exact_accuracy: 0.9965
evaluated_pairs: 285


,pair_id,id,text,label,prediction,prob_a_is_pun
0,1.6,1.6.H,Você sabe qual é a montanha mais limpa de toda...,1,1,0.854342
1,1.6,1.6.N,Você sabe qual é a montanha mais alta de todas...,0,0,0.854342
2,2.6,2.6.H,Por que o bombeiro não gosta de andar? Porque ...,1,1,0.599898
3,2.6,2.6.N,Por que o bombeiro não gosta de andar? Porque ...,0,0,0.599898
4,3.6,3.6.H,Por que o policial não gosta de sabão? Porque ...,1,1,0.684840


In [18]:
test_predictions_df = predict_items_from_pair_model(
    pair_dataframe=test_pairs_df,
    feature_extractor=feature_extractor,
    pairwise_model=pairwise_ensemble_model,
    threshold=best_pair_threshold
)

test_metrics = compute_metrics(
    test_predictions_df["label"],
    test_predictions_df["prediction"]
)

test_metrics.update(
    compute_pair_metrics(test_predictions_df)
)

print_metrics("Test results - pairwise ensemble", test_metrics)

display(test_predictions_df.head())

Test results - pairwise ensemble
accuracy: 0.9982
precision_macro: 0.9982
recall_macro: 0.9982
f1_macro: 0.9982
precision_weighted: 0.9982
recall_weighted: 0.9982
f1_weighted: 0.9982
pair_exact_accuracy: 0.9982
evaluated_pairs: 570


,pair_id,id,text,label,prediction,prob_a_is_pun
0,1.2,1.2.H,Nós chamamos de meio ambiente porque já destru...,1,1,0.605003
1,1.2,1.2.N,Nós chamamos de meio ambiente porque já destru...,0,0,0.605003
2,1.3,1.3.H,"Se um pato perde a pata, ele fica manco ou viúvo?",1,1,0.682465
3,1.3,1.3.N,"Se um pato perde a esposa, ele fica manco ou v...",0,0,0.682465
4,2.1,2.1.H,Sabe como chama a sorveteria do Michel Teló? I...,1,1,0.449788


In [19]:
print("=== Classification report: pairwise ensemble ===")
print(
    classification_report(
        test_predictions_df["label"],
        test_predictions_df["prediction"],
        labels=[0, 1],
        target_names=["0", "1"],
        digits=2,
        zero_division=0
    )
)

=== Classification report: pairwise ensemble ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       570
           1       1.00      1.00      1.00       570

    accuracy                           1.00      1140
   macro avg       1.00      1.00      1.00      1140
weighted avg       1.00      1.00      1.00      1140



In [20]:
cm = confusion_matrix(
    test_predictions_df["label"],
    test_predictions_df["prediction"],
    labels=[0, 1]
)

cm_df = pd.DataFrame(
    cm,
    index=["true_0", "true_1"],
    columns=["pred_0", "pred_1"]
)

display(cm_df)

,pred_0,pred_1
true_0,569,1
true_1,1,569
